# 🚀 02. Train Vision-Only Student Model (Phase 2 - CMKD)

> **Project:** Durian Ripeness Classification via Cross-Modal Knowledge Distillation (CMKD)  
> **Lead:** Leader / Member 1 (Vision Lead & Multi-Modal Framework)  
> **Reference:** `specification/project_specification.pdf` (Section 1.3.3) & `tmp_student.md`  
> **Notebook Role:**
1. Mount Google Drive & verify GPU availability.
2. `git clone` / `git pull` the latest source code from GitHub.
3. Link dataset (`data/final`), Teacher checkpoint (`checkpoints/teachermodal/best_teacher.pth`), and create target folders (`checkpoints/studentmodal`, `results_student`) on Google Drive.
4. Launch 2-phase distillation training (`train_student.py`) and export the benchmark comparison table.

In [20]:
# ==============================================================================
# 1. MOUNT GOOGLE DRIVE & CHECK GPU
# ==============================================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    print("✅ Google Drive mounted successfully!")
else:
    print("ℹ️ Running in Local environment.")

!nvidia-smi

✅ Google Drive mounted successfully!
Fri Nov 20 21:12:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------

In [21]:
# ==============================================================================
# 2. SYNC LATEST CODE FROM GITHUB
# ==============================================================================
REPO_URL = "https://github.com/Cocobaut/durian-ripeness-classification.git"
BRANCH = "cocobaut"
REPO_DIR = "/content/durian-ripeness-classification"

if IN_COLAB:
    %cd /content
    if not os.path.exists(REPO_DIR):
        if BRANCH:
            !git clone -b $BRANCH $REPO_URL $REPO_DIR
        else:
            !git clone $REPO_URL $REPO_DIR
        %cd $REPO_DIR
    else:
        %cd $REPO_DIR
        # Remove temporary repo symlinks (without touching Drive files) to prevent git pull conflicts
        !find . -maxdepth 5 -type l -delete
        !git fetch origin
        if BRANCH:
            !git checkout -f $BRANCH
            !git reset --hard origin/$BRANCH
        else:
            !git reset --hard origin/$(git rev-parse --abbrev-ref HEAD)
else:
    cur = os.path.abspath(".")
    while not (os.path.isdir(os.path.join(cur, "src")) and os.path.isdir(os.path.join(cur, "workspaces"))):
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    os.chdir(cur)

print(f"✅ Working in project root directory: {os.getcwd()}")

/content
/content/durian-ripeness-classification
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 38 (delta 20), reused 38 (delta 20), pack-reused 0 (from 0)
Unpacking objects: 100% (38/38), 462.63 KiB | 12.50 MiB/s, done.
From https://github.com/Cocobaut/durian-ripeness-classification
   194f4f5..16be4c5  cocobaut   -> origin/cocobaut
Already on 'cocobaut'
Your branch is behind 'origin/cocobaut' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
HEAD is now at 16be4c5 [ADD] update v3 student
✅ Working in project root directory: /content/durian-ripeness-classification


In [22]:
# ==============================================================================
# 3. FAST DEPENDENCY INSTALLATION WITH `uv`
# ==============================================================================
!pip install -q uv
!uv pip install --system -r requirements.txt
print("✅ Installed all dependencies successfully!")

Using Python 3.13.16 environment at: /usr
Checked 22 packages in 102ms
✅ Installed all dependencies successfully!


In [23]:
# ==============================================================================
# 4. CONFIGURE CHECKPOINT & DATASET LINKS FROM GOOGLE DRIVE
# ==============================================================================
from pathlib import Path

if IN_COLAB:
    DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/durian-ripeness-cmkd")
    DRIVE_CHECKPOINTS = DRIVE_PROJECT_ROOT / "checkpoints"
    DRIVE_DATA = DRIVE_PROJECT_ROOT / "data" / "final"
    DRIVE_RESULTS_TEACHER = DRIVE_PROJECT_ROOT / "results_teacher"
    DRIVE_RESULTS_STUDENT = DRIVE_PROJECT_ROOT / "results_student"

    # Pre-create studentmodal & results_student folders on Google Drive for real-time persistence
    (DRIVE_CHECKPOINTS / "teachermodal").mkdir(parents=True, exist_ok=True)
    (DRIVE_CHECKPOINTS / "studentmodal").mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTS_STUDENT.mkdir(parents=True, exist_ok=True)

    # Symlink repo directories to Google Drive so outputs persist across Colab disconnects
    !rm -rf checkpoints/unimodal checkpoints/teachermodal checkpoints/studentmodal
    !rm -rf workspaces/member1_vision_lead/04_train_teacher_and_student/results_student
    !mkdir -p checkpoints data/final
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'unimodal')}" checkpoints/unimodal
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'teachermodal')}" checkpoints/teachermodal
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'studentmodal')}" checkpoints/studentmodal
    !ln -sfn "{str(DRIVE_RESULTS_STUDENT)}" workspaces/member1_vision_lead/04_train_teacher_and_student/results_student
    if DRIVE_RESULTS_TEACHER.exists():
        !rm -rf workspaces/member1_vision_lead/04_train_teacher_and_student/results
        !ln -sfn "{str(DRIVE_RESULTS_TEACHER)}" workspaces/member1_vision_lead/04_train_teacher_and_student/results
    for split_name in ["train", "valid", "test"]:
        src_split = DRIVE_DATA / split_name
        if src_split.exists():
            !rm -rf data/final/{split_name}
            !ln -sfn "{str(src_split)}" data/final/{split_name}

Path("checkpoints/studentmodal").mkdir(parents=True, exist_ok=True)
print("✅ Verifying configured paths:")
print(" - Teacher ckpt   :", Path("checkpoints/teachermodal/best_teacher.pth").exists(), "->", Path("checkpoints/teachermodal").resolve())
print(" - Studentmodal   :", Path("checkpoints/studentmodal").resolve())
print(" - Results student:", Path("workspaces/member1_vision_lead/04_train_teacher_and_student/results_student").resolve())
print(" - Manifest CSV   :", Path("data/final/final_multimodal_manifest.csv").exists())

✅ Verifying configured paths:
 - Teacher ckpt   : True -> /content/drive/MyDrive/durian-ripeness-cmkd/checkpoints/teachermodal
 - Studentmodal   : /content/drive/MyDrive/durian-ripeness-cmkd/checkpoints/studentmodal
 - Results student: /content/drive/MyDrive/durian-ripeness-cmkd/results_student
 - Manifest CSV   : True


In [24]:
# ==============================================================================
# 5. TRAIN VISION-ONLY STUDENT (SEED 42) VIA CMKD v4 ON TRAIN + VAL
# ==============================================================================
# CMKD v4 fixes all 4 root causes identified in v3 (where Student dropped below Unimodal Vision):
#  1. --dedup-fruits: 1 sample per unique RGB fruit per epoch (fixes audio-multiplied class bias &
#     flipped class weights Unripe 0.867 / Overripe 1.123 -> true Unripe 1.06 / Overripe 0.94).
#  2. --loss-type ce --label-smoothing 0.08: aligns with Unimodal Vision (removes aggressive
#     ordinal cost_weight=0.5 that suppressed Unripe and shifted boundary fruits).
#  3. --feat-target fused --feat-loss-type relational + --kd-weighting teacher_confidence +
#     --beta 0.5 --gamma 0.25 --temperature 3.0: transfers true 3-modality relational geometry
#     and smoothly down-weights uncertain/wrong Teacher predictions.
#  4. --mixup-alpha 0.2 --mixup-prob 0.5 + --l2-sp-weight 1e-3 + --epochs 18 --warmup-epochs 3:
#     Feature-Space Mixup + L2-SP anchor to Unimodal Vision + shorter schedule prevent overfitting.
# Single seed 42 matches Unimodal Vision & Teacher (~10-12 min on A100).
SEED = 42
!python workspaces/member1_vision_lead/04_train_teacher_and_student/train_student.py \
    --manifest data/final/final_multimodal_manifest.csv \
    --data-dir data/final \
    --teacher-ckpt checkpoints/teachermodal/best_teacher.pth \
    --student-backbone convnextv2_base \
    --save-path checkpoints/studentmodal/best_student.pth \
    --results-dir workspaces/member1_vision_lead/04_train_teacher_and_student/results_student \
    --train-splits train,valid \
    --dedup-fruits \
    --loss-type ce --label-smoothing 0.08 \
    --alpha 1.0 --beta 0.5 --gamma 0.25 --temperature 3.0 \
    --feat-target fused --feat-loss-type relational --kd-weighting teacher_confidence \
    --mixup-alpha 0.2 --mixup-prob 0.5 \
    --l2-sp-weight 1e-3 \
    --batch-size 4 --accum-steps 2 \
    --epochs 18 --warmup-epochs 3 \
    --lr 3e-4 --lr-backbone 5e-6 --min-lr 1e-6 \
    --ema-decay 0.985 \
    --select-metric last \
    --eval-test \
    --fresh \
    --seed {SEED} \
    --device cuda


============================== SEED 42 ==============================
[2026-11-20 21:12:32] ==============================================================================
[2026-11-20 21:12:32] TRAINING VISION-ONLY STUDENT VIA CMKD (STAGE 2)
[2026-11-20 21:12:32] Device: cuda | AMP: True (torch.bfloat16) | batch 4 x accum 2
[2026-11-20 21:12:32] Student backbone : convnextv2_base.fcmae_ft_in1k
[2026-11-20 21:12:32] Distill weights  : alpha=1.0 (CE) | beta=1.0 (KD, T=4.0) | gamma=0.3 (Feat: cosine_relational)
[2026-11-20 21:12:32] Best student ckpt: /content/durian-ripeness-classification/checkpoints/studentmodal/student_trainval_s42.pth
[2026-11-20 21:12:32] ==============================================================================
[2026-11-20 21:12:32] Loading Frozen Teacher from: /content/durian-ripeness-classification/checkpoints/teachermodal/best_teacher.pth
[2026-11-20 21:12:34] L_feat target: Teacher 'vision' feature (256-d) | KD weighting: teacher_correct | EMA: 0.99 | TTA: 

In [25]:
# ==============================================================================
# 6. INDEPENDENT TEST EVALUATION & BENCHMARK COMPARISON TABLE
# ==============================================================================
RES = Path("workspaces/member1_vision_lead/04_train_teacher_and_student/results_student")

!python workspaces/member1_vision_lead/04_train_teacher_and_student/eval_student.py \
    --checkpoint checkpoints/studentmodal/best_student.pth \
    --manifest data/final/final_multimodal_manifest.csv \
    --data-dir data/final \
    --split test \
    --seed {SEED} \
    --output-dir {str(RES)}

comp_md = RES / "benchmark_comparison.md"
if comp_md.is_file():
    print("\n" + comp_md.read_text(encoding="utf-8"))
print("✅ All Student evaluation results saved at:", RES.resolve())

[2026-11-20 22:53:34] [INFO] [eval_student]: Ensembling 3 Students: ['checkpoints/studentmodal/student_trainval_s42.pth', 'checkpoints/studentmodal/student_trainval_s43.pth', 'checkpoints/studentmodal/student_trainval_s44.pth']
[2026-11-20 22:53:34] [INFO] [eval_student]: Student evaluation on 'test' | TTA: True
[2026-11-20 22:53:34] [INFO] [eval_teacher]: Initialized TeacherEvaluator on device: cuda (input_mode=vision)
[2026-11-20 22:53:34] [INFO] [eval_teacher]: Evaluation results will be saved to: /content/durian-ripeness-classification/workspaces/member1_vision_lead/04_train_teacher_and_student/results_student
[2026-11-20 22:53:34] [INFO] [eval_teacher]: === STARTING FULL TEACHER EVALUATION PIPELINE ===
[2026-11-20 22:53:34] [INFO] [eval_teacher]: model=vision_student | split=test | input_mode=vision | strategy=mean_prob
[2026-11-20 22:53:34] [INFO] [eval_teacher]: Starting sample-level evaluation (mask_modality=None)...
[2026-11-20 22:54:00] [INFO] [eval_teacher]: Evaluated 24 sam